# 01 — Data Collection

---

This notebook downloads raw football data for the **German Bundesliga** using the `soccerdata` library.

## Objectives

The purpose of this notebook is to:

- collect raw data from selected providers,
- inspect data availability across the selected seasons,
- save the downloaded tables into the project structure,
- prepare a clean foundation for later match-level dataset construction.

At this stage, the project is focused only on the **Bundesliga**, which is also the target competition for the predictive models developed later in the workflow.

## Selected scope

- **League:** Bundesliga
- **Seasons:** 2023, 2024, 2025

## Expected output

At the end of this notebook, we should have:

- raw data stored in `data/raw/`,
- a summary of successful and failed downloads,
- an overview of table structure and missingness,
- a clear idea of which sources can be used in the next step.

---

## 1. Imports and project setup

In this section, we import all required libraries, make the project source folder available, and load the project configuration.

In [1]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import numpy as np
import soccerdata as sd

# Allow imports from the project root when the notebook is run from /notebooks
PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import (
    RAW_DATA_DIR,
    LEAGUES,
    SEASONS,
    TARGET_LEAGUE,
)
from src.utils import ensure_directories
from src.data_utils import (
    preview_df,
    save_raw,
    log_result,
    missing_overview,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

[04/21/26 22:19:12] INFO     No custom team name replacements found. You can configure these in       ]8;id=13298868;file://C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=13298869;file://C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_config.py#92\92]8;;\
                             C:\Users\cerve\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=13298875;file://C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=13298876;file://C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_config.py#190\190]8;;\
                             C:\Users\cerve\soccerdata\config\league_dict.json.                                    

---

## 2. Create output directories

Before downloading any data, we ensure that the raw data directory exists.
This keeps the workflow reproducible and avoids manual folder creation.

In [2]:
ensure_directories([RAW_DATA_DIR])

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Target league for later modeling: {TARGET_LEAGUE}")
print(f"Leagues: {LEAGUES}")
print(f"Seasons: {SEASONS}")

Project root: C:\Users\cerve\Desktop\DP\match_prediction
Raw data directory: C:\Users\cerve\Desktop\DP\match_prediction\data\raw
Target league for later modeling: GER-Bundesliga
Leagues: ['GER-Bundesliga']
Seasons: [2023, 2024, 2025]


---

## 3. General download log

To keep track of the whole data collection process, we use a simple download log.
Each attempted table download will be recorded as either a success or a failure.

In [3]:
download_log = []
understat_data = {}
fbref_data = {}

---

## 4. Understat data collection

`Understat` is the main source of expected metrics in this project.

At this stage, we attempt to download the following tables:

- match schedule,
- team match statistics,
- player season statistics.

The shot-level event table is intentionally skipped in this first version because it is computationally heavy and not required for building the initial match-level dataset.

Because some endpoints may fail or take longer depending on provider availability, each download is handled separately.

In [4]:
understat = sd.Understat(
    leagues=LEAGUES,
    seasons=SEASONS,
    no_cache=True,
)

                    INFO     Saving cached data to C:\Users\cerve\soccerdata\data\Understat          ]8;id=13298883;file://C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=13298884;file://C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-04-21 22:19:13] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


[04/21/26 22:19:13] INFO     Successfully loaded TLS library:                                      ]8;id=13298891;file://C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=13298892;file://C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python                 
                             .3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages                 
                             \tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll                             

---

### 4.1 Understat schedule

The schedule table should contain match-level information such as teams, dates, scores, and expected goals.
This is one of the most important raw inputs for later dataset construction.

In [5]:
try:
    df_schedule = understat.read_schedule()
    understat_data["schedule"] = df_schedule

    save_path = save_raw(
        df=df_schedule,
        raw_data_dir=RAW_DATA_DIR,
        provider="understat",
        table_name="schedule",
    )

    log_result(
        results=download_log,
        provider="understat",
        table_name="schedule",
        status="success",
        shape=df_schedule.shape,
        note=str(save_path),
    )

    preview_df(df_schedule, "Understat — schedule")

except Exception as e:
    log_result(
        results=download_log,
        provider="understat",
        table_name="schedule",
        status="failed",
        note=str(e),
    )
    print(f"Understat schedule failed: {e}")

Understat — schedule
Shape: (918, 17)
Columns (17): ['league_id', 'season_id', 'game_id', 'date', 'home_team_id', 'away_team_id', 'home_team', 'away_team', 'away_team_code', 'home_team_code', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'is_result', 'has_data', 'url']
                                                                         league_id  season_id  game_id                date  home_team_id  \
league         season game                                                                                                                 
GER-Bundesliga 2324   2023-08-18 Werder Bremen-Bayern Munich                     3       2023    23065 2023-08-18 18:30:00           123   
                      2023-08-19 Augsburg-Borussia M.Gladbach                    3       2023    23069 2023-08-19 13:30:00           121   
                      2023-08-19 Bayer Leverkusen-RasenBallsport Leipzig         3       2023    23066 2023-08-19 13:30:00           119   
                      2023

---

### 4.2 Understat team match statistics

This table may provide team-level match statistics that can later be transformed into rolling pre-match features.

In [6]:
try:
    df_team_match_stats = understat.read_team_match_stats()
    understat_data["team_match_stats"] = df_team_match_stats

    save_path = save_raw(
        df=df_team_match_stats,
        raw_data_dir=RAW_DATA_DIR,
        provider="understat",
        table_name="team_match_stats",
    )

    log_result(
        results=download_log,
        provider="understat",
        table_name="team_match_stats",
        status="success",
        shape=df_team_match_stats.shape,
        note=str(save_path),
    )

    preview_df(df_team_match_stats, "Understat — team_match_stats")

except Exception as e:
    log_result(
        results=download_log,
        provider="understat",
        table_name="team_match_stats",
        status="failed",
        note=str(e),
    )
    print(f"Understat team match stats failed: {e}")

Understat — team_match_stats
Shape: (882, 26)
Columns (26): ['league_id', 'season_id', 'game_id', 'date', 'home_team_id', 'away_team_id', 'home_team', 'away_team', 'away_team_code', 'home_team_code', 'away_points', 'away_expected_points', 'away_goals', 'away_xg', 'away_np_xg', 'away_np_xg_difference', 'away_ppda', 'away_deep_completions', 'home_points', 'home_expected_points', 'home_goals', 'home_xg', 'home_np_xg', 'home_np_xg_difference', 'home_ppda', 'home_deep_completions']
                                                                         league_id  season_id  game_id                date  home_team_id  \
league         season game                                                                                                                 
GER-Bundesliga 2324   2023-08-18 Werder Bremen-Bayern Munich                     3       2023    23065 2023-08-18 18:30:00           123   
                      2023-08-19 Augsburg-Borussia M.Gladbach                    3       2023    2

---

### 4.3 Understat player season statistics

Player-level season statistics are not needed immediately for the first match-level dataset, but they may become useful later when adding player quality or squad-strength information.

In [7]:
try:
    df_player_season_stats = understat.read_player_season_stats()
    understat_data["player_season_stats"] = df_player_season_stats

    save_path = save_raw(
        df=df_player_season_stats,
        raw_data_dir=RAW_DATA_DIR,
        provider="understat",
        table_name="player_season_stats",
    )

    log_result(
        results=download_log,
        provider="understat",
        table_name="player_season_stats",
        status="success",
        shape=df_player_season_stats.shape,
        note=str(save_path),
    )

    preview_df(df_player_season_stats, "Understat — player_season_stats")

except Exception as e:
    log_result(
        results=download_log,
        provider="understat",
        table_name="player_season_stats",
        status="failed",
        note=str(e),
    )
    print(f"Understat player season stats failed: {e}")

Understat — player_season_stats
Shape: (1464, 19)
Columns (19): ['league_id', 'season_id', 'team_id', 'player_id', 'position', 'matches', 'minutes', 'goals', 'xg', 'np_goals', 'np_xg', 'assists', 'xa', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'xg_chain', 'xg_buildup']
                                                 league_id  season_id  team_id  player_id position  matches  minutes  goals         xg  \
league         season team     player                                                                                                    
GER-Bundesliga 2324   Augsburg Arne Engels               3       2023      121      11323    D M S       32     1386      3   1.373717   
                               Arne Maier                3       2023      121       5298      M S       22      981      2   1.195101   
                               Dion Beljo                3       2023      121      11324    F M S       26      506      2   3.422512   
                              

---

## 5. FBref data collection

`FBref` is used in this notebook as a complementary data provider for match schedule and team-level seasonal statistics.

At this stage, the goal is not to build final features yet, but rather to:

- identify which tables are available through the current `soccerdata` interface,
- save them in raw form,
- inspect their structure,
- and evaluate their potential usefulness for later Bundesliga match prediction.

For the current setup, we focus on the following supported tables:

- schedule / results,
- team season standard stats,
- team season shooting stats,
- team season playing-time stats,
- team season miscellaneous stats.

These tables may later help us construct season-level context variables, validate cross-provider consistency, and enrich the Bundesliga dataset with additional team information.

In [8]:
fbref = sd.FBref(
    leagues=LEAGUES,
    seasons=SEASONS,
    no_cache=True,
)

[04/21/26 22:19:17] INFO     Saving cached data to C:\Users\cerve\soccerdata\data\FBref              ]8;id=13298897;file://C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=13298898;file://C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_common.py#250\250]8;;\

----

### 5.1 FBref schedule

This table should contain competition schedule and match result information that may be useful as a secondary match-level source.

In [9]:
try:
    df_fbref_schedule = fbref.read_schedule()
    fbref_data["schedule"] = df_fbref_schedule

    save_path = save_raw(
        df=df_fbref_schedule,
        raw_data_dir=RAW_DATA_DIR,
        provider="fbref",
        table_name="schedule",
    )

    log_result(
        results=download_log,
        provider="fbref",
        table_name="schedule",
        status="success",
        shape=df_fbref_schedule.shape,
        note=str(save_path),
    )

    preview_df(df_fbref_schedule, "FBref — schedule")

except Exception as e:
    log_result(
        results=download_log,
        provider="fbref",
        table_name="schedule",
        status="failed",
        note=str(e),
    )
    print(f"FBref schedule failed: {e}")

[04/21/26 22:19:56] ERROR    Error while scraping https://fbref.com/en/comps/. Retrying in 0         ]8;id=13298904;file://C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=13298905;file://C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\soccerdata\_common.py#623\623]8;;\
                             seconds... (attempt 1 of 5).                                                          
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.               
                             3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\so               
                             ccerdata\_common.py", line 607, in _download_and_save                                 
                                 response = self._validate_page(url).encode("utf-8")                               
                                            ^^^^^^^^^^^^^^^^^^^^^^^^                                               
                               File                                                                                
                             "C:\Users\cerve\AppData\Local\Packages\PythonSoftwareFoundation.Python.               
                             3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\so               
                             ccerdata\fbref.py", line 1031, in _validate_page                                      
                                 raise Exception(                                                                  
                             Exception: Could not retrieve page content within timeout. Possible                   
                             reasons: failed CAPTCHA, IP block or network issues.                                  

FBref — schedule
Shape: (922, 14)
Columns (14): ['round', 'week', 'day', 'date', 'time', 'home_team', 'score', 'away_team', 'attendance', 'venue', 'referee', 'match_report', 'notes', 'game_id']
                                                                   round  week  day       date   time      home_team score      away_team  \
league         season game                                                                                                                  
GER-Bundesliga 2324   2023-08-18 Werder Bremen-Bayern Munich  Bundesliga     1  Fri 2023-08-18  20:30  Werder Bremen   0–4  Bayern Munich   
                      2023-08-19 Augsburg-Gladbach            Bundesliga     1  Sat 2023-08-19  15:30       Augsburg   4–4       Gladbach   
                      2023-08-19 Dortmund-Köln                Bundesliga     1  Sat 2023-08-19  18:30       Dortmund   1–0           Köln   
                      2023-08-19 Hoffenheim-Freiburg          Bundesliga     1  Sat 2023-08-19  15:30

---

### 5.2 FBref team season stats — standard

Standard team-level season statistics may include useful descriptive summaries and can later help with cross-checking provider consistency.

In [10]:
try:
    df_fbref_team_standard = fbref.read_team_season_stats(stat_type="standard")
    fbref_data["team_season_standard"] = df_fbref_team_standard

    save_path = save_raw(
        df=df_fbref_team_standard,
        raw_data_dir=RAW_DATA_DIR,
        provider="fbref",
        table_name="team_season_standard",
    )

    log_result(
        results=download_log,
        provider="fbref",
        table_name="team_season_standard",
        status="success",
        shape=df_fbref_team_standard.shape,
        note=str(save_path),
    )

    preview_df(df_fbref_team_standard, "FBref — team_season_standard")

except Exception as e:
    log_result(
        results=download_log,
        provider="fbref",
        table_name="team_season_standard",
        status="failed",
        note=str(e),
    )
    print(f"FBref team season standard failed: {e}")

FBref — team_season_standard
Shape: (54, 21)
Columns (21): [('players_used', ''), ('Age', ''), ('Poss', ''), ('Playing Time', 'MP'), ('Playing Time', 'Starts'), ('Playing Time', 'Min'), ('Playing Time', '90s'), ('Performance', 'Gls'), ('Performance', 'Ast'), ('Performance', 'G+A'), ('Performance', 'G-PK'), ('Performance', 'PK'), ('Performance', 'PKatt'), ('Performance', 'CrdY'), ('Performance', 'CrdR'), ('Per 90 Minutes', 'Gls'), ('Per 90 Minutes', 'Ast'), ('Per 90 Minutes', 'G+A'), ('Per 90 Minutes', 'G-PK'), ('Per 90 Minutes', 'G+A-PK'), ('url', '')]
                                    players_used   Age  Poss Playing Time                  Performance                                   \
                                                                       MP Starts   Min 90s         Gls Ast  G+A G-PK PK PKatt CrdY CrdR   
league         season team                                                                                                                
GER-Bundesliga 2324   Au

---

### 5.3 FBref team season stats — shooting

The shooting table is especially relevant because it may contain xG-related aggregates and shot-based attacking summaries.

In [11]:
try:
    df_fbref_team_shooting = fbref.read_team_season_stats(stat_type="shooting")
    fbref_data["team_season_shooting"] = df_fbref_team_shooting

    save_path = save_raw(
        df=df_fbref_team_shooting,
        raw_data_dir=RAW_DATA_DIR,
        provider="fbref",
        table_name="team_season_shooting",
    )

    log_result(
        results=download_log,
        provider="fbref",
        table_name="team_season_shooting",
        status="success",
        shape=df_fbref_team_shooting.shape,
        note=str(save_path),
    )

    preview_df(df_fbref_team_shooting, "FBref — team_season_shooting")

except Exception as e:
    log_result(
        results=download_log,
        provider="fbref",
        table_name="team_season_shooting",
        status="failed",
        note=str(e),
    )
    print(f"FBref team season shooting failed: {e}")

FBref — team_season_shooting
Shape: (54, 13)
Columns (13): [('players_used', ''), ('90s', ''), ('Standard', 'Gls'), ('Standard', 'Sh'), ('Standard', 'SoT'), ('Standard', 'SoT%'), ('Standard', 'Sh/90'), ('Standard', 'SoT/90'), ('Standard', 'G/Sh'), ('Standard', 'G/SoT'), ('Standard', 'PK'), ('Standard', 'PKatt'), ('url', '')]
                                    players_used 90s Standard                                                     \
                                                          Gls   Sh  SoT  SoT%  Sh/90 SoT/90  G/Sh G/SoT PK PKatt   
league         season team                                                                                         
GER-Bundesliga 2324   Augsburg                29  34       49  429  123  28.7  12.62   3.62   0.1  0.36  5     7   
                      Bayern Munich           30  34       93  633  229  36.2  18.62   6.74  0.14  0.38  5     5   
                      Bochum                  25  34       41  524  145  27.7  15.41   4.26  

---

### 5.4 FBref team season stats — playing time

The playing-time table may later provide useful information about squad usage, rotation, and season-level team context.

In [12]:
try:
    df_fbref_team_playing_time = fbref.read_team_season_stats(stat_type="playing_time")
    fbref_data["team_season_playing_time"] = df_fbref_team_playing_time

    save_path = save_raw(
        df=df_fbref_team_playing_time,
        raw_data_dir=RAW_DATA_DIR,
        provider="fbref",
        table_name="team_season_playing_time",
    )

    log_result(
        results=download_log,
        provider="fbref",
        table_name="team_season_playing_time",
        status="success",
        shape=df_fbref_team_playing_time.shape,
        note=str(save_path),
    )

    preview_df(df_fbref_team_playing_time, "FBref — team_season_playing_time")

except Exception as e:
    log_result(
        results=download_log,
        provider="fbref",
        table_name="team_season_playing_time",
        status="failed",
        note=str(e),
    )
    print(f"FBref team season playing_time failed: {e}")

FBref — team_season_playing_time
Shape: (54, 19)
Columns (19): [('players_used', ''), ('Age', ''), ('Playing Time', 'MP'), ('Playing Time', 'Min'), ('Playing Time', 'Mn/MP'), ('Playing Time', 'Min%'), ('Playing Time', '90s'), ('Starts', 'Starts'), ('Starts', 'Mn/Start'), ('Starts', 'Compl'), ('Subs', 'Subs'), ('Subs', 'Mn/Sub'), ('Subs', 'unSub'), ('Team Success', 'PPM'), ('Team Success', 'onG'), ('Team Success', 'onGA'), ('Team Success', '+/-'), ('Team Success', '+/-90'), ('url', '')]
                                    players_used   Age Playing Time                      Starts                Subs               \
                                                                 MP   Min Mn/MP Min% 90s Starts Mn/Start Compl Subs Mn/Sub unSub   
league         season team                                                                                                         
GER-Bundesliga 2324   Augsburg                29  26.0           34  3060    90  100  34    374       81   214  1

---

### 5.5 FBref team season stats — miscellaneous

The miscellaneous table may contain additional team-level information that could become useful later as supporting contextual variables.

In [13]:
try:
    df_fbref_team_misc = fbref.read_team_season_stats(stat_type="misc")
    fbref_data["team_season_misc"] = df_fbref_team_misc

    save_path = save_raw(
        df=df_fbref_team_misc,
        raw_data_dir=RAW_DATA_DIR,
        provider="fbref",
        table_name="team_season_misc",
    )

    log_result(
        results=download_log,
        provider="fbref",
        table_name="team_season_misc",
        status="success",
        shape=df_fbref_team_misc.shape,
        note=str(save_path),
    )

    preview_df(df_fbref_team_misc, "FBref — team_season_misc")

except Exception as e:
    log_result(
        results=download_log,
        provider="fbref",
        table_name="team_season_misc",
        status="failed",
        note=str(e),
    )
    print(f"FBref team season misc failed: {e}")

FBref — team_season_misc
Shape: (54, 15)
Columns (15): [('players_used', ''), ('90s', ''), ('Performance', 'CrdY'), ('Performance', 'CrdR'), ('Performance', '2CrdY'), ('Performance', 'Fls'), ('Performance', 'Fld'), ('Performance', 'Off'), ('Performance', 'Crs'), ('Performance', 'Int'), ('Performance', 'TklW'), ('Performance', 'PKwon'), ('Performance', 'PKcon'), ('Performance', 'OG'), ('url', '')]
                                    players_used 90s Performance                                                         \
                                                            CrdY CrdR 2CrdY  Fls  Fld Off  Crs  Int TklW PKwon PKcon OG   
league         season team                                                                                                
GER-Bundesliga 2324   Augsburg                29  34          71    3     1  454  338  81  631  295  319     5    10  2   
                      Bayern Munich           30  34          47    2     1  304  306  60  655  288  323    

---

## 6. Download summary

We now summarize which tables were downloaded successfully and which ones failed.
This is useful both for debugging and for documenting the data collection phase.

In [14]:
download_summary = pd.DataFrame(download_log)
download_summary

,provider,table_name,status,n_rows,n_cols,note
0,understat,schedule,success,918,17,C:\Users\cerve\Desktop\DP\match_prediction\dat...
1,understat,team_match_stats,success,882,26,C:\Users\cerve\Desktop\DP\match_prediction\dat...
2,understat,player_season_stats,success,1464,19,C:\Users\cerve\Desktop\DP\match_prediction\dat...
3,fbref,schedule,success,922,14,C:\Users\cerve\Desktop\DP\match_prediction\dat...
4,fbref,team_season_standard,success,54,21,C:\Users\cerve\Desktop\DP\match_prediction\dat...
5,fbref,team_season_shooting,success,54,13,C:\Users\cerve\Desktop\DP\match_prediction\dat...
6,fbref,team_season_playing_time,success,54,19,C:\Users\cerve\Desktop\DP\match_prediction\dat...
7,fbref,team_season_misc,success,54,15,C:\Users\cerve\Desktop\DP\match_prediction\dat...


A quick frequency table makes it easier to see the overall outcome of the collection process.

In [15]:
download_summary["status"].value_counts().to_frame("count")

,count
status,
success,8


---

## 7. Quick inspection of league and season coverage

Before moving on, it is useful to verify whether the downloaded tables contain the expected leagues and seasons.
At this point, we are not cleaning the data yet — only checking coverage.

In [16]:
for name, df in understat_data.items():
    print(f"\nUNDERSTAT — {name}")
    if "league" in df.columns:
        print("Leagues:", sorted(df["league"].dropna().astype(str).unique().tolist()))
    if "season" in df.columns:
        print("Seasons:", sorted(df["season"].dropna().unique().tolist()))


UNDERSTAT — schedule

UNDERSTAT — team_match_stats

UNDERSTAT — player_season_stats


In [17]:
for name, df in fbref_data.items():
    print(f"\nFBREF — {name}")
    if "league" in df.columns:
        print("Leagues:", sorted(df["league"].dropna().astype(str).unique().tolist()))
    if "season" in df.columns:
        print("Seasons:", sorted(df["season"].dropna().unique().tolist()))


FBREF — schedule

FBREF — team_season_standard

FBREF — team_season_shooting

FBREF — team_season_playing_time

FBREF — team_season_misc


---

## 8. Missing values overview

A quick missing-value scan helps identify which tables look promising for downstream feature engineering.

The goal here is not to solve missingness yet, but to understand how complete the raw tables are.

In [18]:
for name, df in understat_data.items():
    print(f"\nUNDERSTAT — {name}")
    display(missing_overview(df))


UNDERSTAT — schedule


,missing_share
away_xg,0.039216
home_xg,0.039216
away_goals,0.039216
home_goals,0.039216
league_id,0.000000
home_team_code,0.000000
has_data,0.000000
is_result,0.000000
away_team_code,0.000000
season_id,0.000000



UNDERSTAT — team_match_stats


,missing_share
home_ppda,0.001134
away_ppda,0.001134
league_id,0.000000
season_id,0.000000
home_np_xg_difference,0.000000
home_np_xg,0.000000
home_xg,0.000000
home_goals,0.000000
home_expected_points,0.000000
home_points,0.000000



UNDERSTAT — player_season_stats


,missing_share
league_id,0.0
np_xg,0.0
xg_chain,0.0
red_cards,0.0
yellow_cards,0.0
key_passes,0.0
shots,0.0
xa,0.0
assists,0.0
np_goals,0.0


In [19]:
for name, df in fbref_data.items():
    print(f"\nFBREF — {name}")
    display(missing_overview(df))


FBREF — schedule


,missing_share
notes,0.994577
round,0.331887
score,0.039046
attendance,0.039046
referee,0.039046
match_report,0.039046
game_id,0.039046
week,0.004338
day,0.000000
date,0.000000



FBREF — team_season_standard


missing_share
players_used                     0.0
Performance    PK                0.0
Per 90 Minutes G+A-PK            0.0
               G-PK              0.0
               G+A               0.0
               Ast               0.0
               Gls               0.0
Performance    CrdR              0.0
               CrdY              0.0
               PKatt             0.0
               G-PK              0.0
Age                              0.0
Performance    G+A               0.0
               Ast               0.0
               Gls               0.0
Playing Time   90s               0.0
               Min               0.0
               Starts            0.0
               MP                0.0
Poss                             0.0


FBREF — team_season_shooting


missing_share
players_used                   0.0
90s                            0.0
Standard     Gls               0.0
             Sh                0.0
             SoT               0.0
             SoT%              0.0
             Sh/90             0.0
             SoT/90            0.0
             G/Sh              0.0
             G/SoT             0.0
             PK                0.0
             PKatt             0.0
url                            0.0


FBREF — team_season_playing_time


missing_share
players_used                     0.0
Subs         Subs                0.0
Team Success +/-90               0.0
             +/-                 0.0
             onGA                0.0
             onG                 0.0
             PPM                 0.0
Subs         unSub               0.0
             Mn/Sub              0.0
Starts       Compl               0.0
Age                              0.0
Starts       Mn/Start            0.0
             Starts              0.0
Playing Time 90s                 0.0
             Min%                0.0
             Mn/MP               0.0
             Min                 0.0
             MP                  0.0
url                              0.0


FBREF — team_season_misc


missing_share
Performance  PKwon       0.333333
             PKcon       0.333333
players_used             0.000000
90s                      0.000000
Performance  CrdY        0.000000
             CrdR        0.000000
             2CrdY       0.000000
             Fls         0.000000
             Fld         0.000000
             Off         0.000000
             Crs         0.000000
             Int         0.000000
             TklW        0.000000
             OG          0.000000
url                      0.000000

---

## 9. Raw files saved

This final check shows the files that were saved into the raw data directory.
It helps confirm that the notebook produced persistent outputs.

In [20]:
raw_files = sorted([str(path.relative_to(PROJECT_ROOT)) for path in RAW_DATA_DIR.rglob("*.parquet")])
pd.DataFrame({"saved_files": raw_files})

,saved_files
0,data\raw\fbref\schedule.parquet
1,data\raw\fbref\team_season_misc.parquet
2,data\raw\fbref\team_season_playing_time.parquet
3,data\raw\fbref\team_season_shooting.parquet
4,data\raw\fbref\team_season_standard.parquet
5,data\raw\understat\player_season_stats.parquet
6,data\raw\understat\schedule.parquet
7,data\raw\understat\team_match_stats.parquet


---

## 10. Conclusions

This notebook established the raw-data layer of the project for the **Bundesliga**.

### Main outcomes

- raw football data were downloaded from selected `soccerdata` providers,
- successfully downloaded tables were saved to `data/raw`,
- table structure, seasonal coverage, and missingness were inspected,
- the project is now ready for the next step: match-level dataset construction.

### Current scope

The project currently works with:

- **League:** Bundesliga
- **Seasons:** 2023, 2024, 2025

### Next step

The next notebook will focus on:

1. selecting the most useful raw tables,
2. harmonizing team names and competition labels,
3. building a unified match-level dataset,
4. defining target variables for both ML and Poisson models.

---

In [21]:
df_schedule.columns

Index(['league_id', 'season_id', 'game_id', 'date', 'home_team_id', 'away_team_id', 'home_team', 'away_team', 'away_team_code',
       'home_team_code', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'is_result', 'has_data', 'url'],
      dtype='object')

In [22]:
df_team_match_stats.columns

Index(['league_id', 'season_id', 'game_id', 'date', 'home_team_id', 'away_team_id', 'home_team', 'away_team', 'away_team_code',
       'home_team_code', 'away_points', 'away_expected_points', 'away_goals', 'away_xg', 'away_np_xg', 'away_np_xg_difference',
       'away_ppda', 'away_deep_completions', 'home_points', 'home_expected_points', 'home_goals', 'home_xg', 'home_np_xg',
       'home_np_xg_difference', 'home_ppda', 'home_deep_completions'],
      dtype='object')

In [23]:
df_schedule.head().T

league                                 GER-Bundesliga                                          \
season                                           2324                                           
game           2023-08-18 Werder Bremen-Bayern Munich 2023-08-19 Augsburg-Borussia M.Gladbach   
league_id                                           3                                       3   
season_id                                        2023                                    2023   
game_id                                         23065                                   23069   
date                              2023-08-18 18:30:00                     2023-08-19 13:30:00   
home_team_id                                      123                                     121   
away_team_id                                      117                                     130   
home_team                               Werder Bremen                                Augsburg   
away_team                               Bayern Munich                     Borussia M.Gladbach   
away_team_code                                    BAY                                     BMG   
home_team_code                                    WER                                     AUG   
home_goals                                          0                                       4   
away_goals                                          4                                       4   
home_xg                                       0.63974                                 2.53482   
away_xg                                       2.89704                                 1.91849   
is_result                                        True                                    True   
has_data                                         True                                    True   
url                 https://understat.com/match/23065       https://understat.com/match/23069   

league                                                                                                     \
season                                                                                                      
game           2023-08-19 Bayer Leverkusen-RasenBallsport Leipzig 2023-08-19 Borussia Dortmund-FC Cologne   
league_id                                                       3                                       3   
season_id                                                    2023                                    2023   
game_id                                                     23066                                   23071   
date                                          2023-08-19 13:30:00                     2023-08-19 16:30:00   
home_team_id                                                  119                                     129   
away_team_id                                                  136                                     134   
home_team                                        Bayer Leverkusen                       Borussia Dortmund   
away_team                                  RasenBallsport Leipzig                              FC Cologne   
away_team_code                                                RBL                                     COL   
home_team_code                                                LEV                                     DOR   
home_goals                                                      3                                       1   
away_goals                                                      2                                       0   
home_xg                                                   1.73279                                 1.72317   
away_xg                                                   1.60393                                 1.51393   
is_result                                                    True                                    True   
has_data                                                     True                                